# IE4422: ScienceQA answer checker and tutor on Colab

Run in **Google Colab with a GPU runtime**. Upload this notebook and the matching `IE4422_Source.zip`.
Your own computer needs no GPU, Python installation or Hugging Face account.

Default: prepare/restore the audited ScienceQA selection and fixed Qwen checkpoint, then run an
**answer-only check on training questions**. One model instance receives each image, question, context and
ordered choices, and returns one option letter. Reference answers enter offline scoring after generation.
This development check does not train the model or establish hint quality. It uses no validation/test items.

The browser tutor and original three-hint demo remain available through `INTERFACE` in step 3.
Those existing demo modes offer train/validation questions; the answer checker is restricted to train.

First download: about 626 MB of data and 7.5 GB of weights. Drive can retain prepared data, weights and results;
allow at least 8 GB free for the checkpoint plus data/results. All project jobs use one project environment
with Colab's existing CUDA wheels; the notebook kernel's packages are not replaced.

This is an exploratory prototype, not a formal assessment or the final A-E research protocol.
The UI's submitted-answer check uses separate references internally; they are not returned by the question API.
**Show inspection report** explicitly reveals reference material, so leave it closed during a student attempt.

**Colab availability:** Google's [FAQ](https://research.google.com/colaboratory/faq.html) restricts interacting primarily
through a web UI on free managed runtimes without a positive compute-unit balance. Use an eligible runtime for the UI;
the notebook demo remains available. Embedding does not exempt a runtime from those restrictions.
Keep the Colab notebook open; this is a session UI, not a persistent or publicly shared service.


## 1. Check the cloud runtime

A local GPU or Python installation is unnecessary. If CUDA is unavailable, select a GPU runtime and reconnect.
Colab does not guarantee a particular GPU or session length.


In [ ]:
import json
import os
from pathlib import Path, PurePosixPath
import shutil
import stat
import subprocess
import sys
import zipfile

if not Path('/content').is_dir() or not sys.platform.startswith('linux'):
    raise RuntimeError('Open this notebook in Google Colab, not a local notebook kernel.')
if sys.version_info < (3, 12):
    raise RuntimeError('Select a current Colab Python 3.12+ runtime for the shared dependency pins.')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime → Change runtime type → GPU, then reconnect and rerun.')
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print('Local disk free (GiB):', round(shutil.disk_usage('/content').free / 2**30, 1))


## 2. Upload and extract the source ZIP

Upload the updated `IE4422_Source.zip`. Leave `UPLOAD_NEW_SOURCE=True` when replacing an older bundle;
set it to False only when deliberately reusing the uploaded ZIP in the same runtime.
Source runs from `/content/IE4422`; all paths below are set automatically.


In [ ]:
WORK = Path('/content/IE4422')
SOURCE_ZIP = Path('/content/IE4422_Source.zip')
UPLOAD_NEW_SOURCE = True  # @param {type:"boolean"}
if UPLOAD_NEW_SOURCE or not SOURCE_ZIP.is_file():
    if globals().get('UI_SERVER') is not None:
        UI_SERVER.stop()  # Release the old worker before replacing its source files.
    from google.colab import files
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('Upload exactly one IE4422 source ZIP.')
    SOURCE_ZIP.write_bytes(uploaded[archives[0]])

# Validate paths before extracting the package. The ZIP contains source only.
with zipfile.ZipFile(SOURCE_ZIP) as source:
    names = set()
    for info in source.infolist():
        path = PurePosixPath(info.filename)
        if (not path.parts or path.parts[0] != 'IE4422' or path.is_absolute()
                or '..' in path.parts or '\\' in info.filename or ':' in info.filename
                or info.filename in names or stat.S_ISLNK(info.external_attr >> 16)
                or not (Path('/content') / info.filename).resolve().is_relative_to(WORK.resolve())):
            raise ValueError(f'Unexpected ZIP member: {info.filename}')
        names.add(info.filename)
    if 'IE4422/bundle_info.json' not in names:
        raise ValueError('Use the supplied IE4422_Source.zip, not a general repository download.')
    source.extractall('/content')
sys.path.insert(0, str(WORK))
import importlib
import notebooks.colab_support as storage
importlib.reload(storage)  # Pick up the helper from the newly uploaded bundle.
from notebooks.colab_support import (read_json, write_json, archive_prepared,
                                    restore_prepared, prepared_signature, prepared_files, run_visible)
bundle = read_json(WORK / 'bundle_info.json')
print('Source extracted:', WORK)
print('Bundle created:', bundle['created_at_utc'])


## 3. Choose workflow and persistence settings

`answer checker` runs the new answer-only check on audited training questions in step 8.
`browser UI` starts the existing interactive tutor; `notebook demo` runs its original three-hint workflow.
Approve mounting **your own** Drive. Work runs from temporary Colab storage; Drive retains prepared data,
optional model weights and results. `REBUILD_DATA=True` repeats preparation.


In [ ]:
INTERFACE = 'answer checker'  # @param ["answer checker", "browser UI", "notebook demo"]
USE_DRIVE = True  # @param {type:"boolean"}
SAVE_MODEL_TO_DRIVE = True  # @param {type:"boolean"}
REBUILD_DATA = False  # @param {type:"boolean"}
PERSISTENT = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSISTENT = Path('/content/drive/MyDrive/IE4422')
    PERSISTENT.mkdir(parents=True, exist_ok=True)
print('Persistent folder:', PERSISTENT or 'disabled; download results at the end')


## 4. Set up one project environment

This may take several minutes on the first run. The helper preserves Colab's installed CUDA torch/torchvision
and respects their active dependency limits, applies compatible shared project pins, checks the project's dependency graph,
and verifies CUDA inside the project environment. Environment creation skips `ensurepip`; Colab's existing pip explicitly targets the project Python.
Each command's output and errors are displayed live. It does not alter the notebook kernel or silently switch to CPU.
Local Windows setup remains unchanged. Actual Colab package versions are saved with each generated result.


In [ ]:
if globals().get('UI_SERVER') is not None:
    UI_SERVER.stop()  # Do not change an environment used by a running worker.
ENVIRONMENT = Path('/content/ie4422_env')
run_visible([sys.executable, str(WORK / 'notebooks/colab_support.py'),
                'setup', str(WORK), str(ENVIRONMENT)], cwd=WORK)
PYTHON = ENVIRONMENT / 'bin/python'
print('Project Python:', PYTHON)


## 5. Check the packaged code

These focused tests use synthetic inputs/fake tutor calls. They check data handling, reference separation,
model reuse, caching and packaging; passing them does not establish model quality or real GPU feasibility.


In [ ]:
run_visible([str(PYTHON), '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=WORK)


## 6. Prepare and finalize the audited ScienceQA selection

`prepare_data.py` downloads and validates the candidates. `finalize_data.py` applies the packaged visual-need decisions and exports only retained questions and images.
The reusable Drive archive includes both preparation and final exports. Its signature binds the source/configuration and saved audit decisions; older or mismatched archives must be rebuilt.

Preparation retains the official splits for the existing project. The answer checker reads only final
training questions and training references; preparing other splits does not evaluate them.


In [ ]:
if globals().get('UI_SERVER') is not None:
    UI_SERVER.stop()  # Do not rebuild exports while the UI is serving them.
signature = prepared_signature(WORK)
local_marker = WORK / 'tmp/prepared_signature.json'
local_archive = WORK / 'tmp/scienceqa_prepared.zip'
exports = [WORK / name for name in prepared_files()]
ready = (not REBUILD_DATA and local_marker.is_file()
         and read_json(local_marker) == signature and all(path.is_file() for path in exports))
if not ready and not REBUILD_DATA and PERSISTENT and (PERSISTENT / local_archive.name).is_file():
    local_archive.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(PERSISTENT / local_archive.name, local_archive)
    ready = restore_prepared(WORK, local_archive)
    print('Restored matching prepared and audited data from Drive.')
if not ready:
    run_visible([str(PYTHON), 'prepare_data.py'], cwd=WORK)
# Always validate the saved decisions and rebuild the audited exports before use.
run_visible([str(PYTHON), 'finalize_data.py'], cwd=WORK)
if not ready:
    archive_prepared(WORK, local_archive)
    if PERSISTENT:
        pending = PERSISTENT / (local_archive.name + '.partial')
        shutil.copy2(local_archive, pending)
        pending.replace(PERSISTENT / local_archive.name)
    print('Saved a reusable archive including the audited final set.')
write_json(local_marker, signature)
audit = read_json(WORK / 'reports/scienceqa/visual_audit.json')
print('Audited final counts:', audit['final_counts'])


## 7. Download/restore the fixed Qwen checkpoint

Drive can retain weights between sessions. A complete checkpoint is copied to local Colab storage before loading.
This cell downloads/restores files. The checker loads them once for its batch; the UI loads on its first request.
No weights are in the source ZIP. Rerun after an interrupted transfer; incomplete checkpoints are rejected.


In [ ]:
model_command = [str(PYTHON), str(WORK / 'notebooks/colab_support.py'), 'model', str(WORK)]
if PERSISTENT and SAVE_MODEL_TO_DRIVE:
    model_command.extend(['--persistent', str(PERSISTENT / 'models')])
run_visible(model_command, cwd=WORK)


## 8. Run the answer-only training check

Runs only when `INTERFACE='answer checker'`. `CHECK_LIMIT=10` checks the first ten audited training
questions in saved order; set it to 0 for the entire audited training set. This is a development sample,
not a held-out evaluation. Validation/test items cannot be selected by this checker.

Each question uses one direct model call, requesting exactly one option letter without an explanation.
Whitespace and letter case are normalized; extra text and out-of-range letters are invalid. Invalid
outputs and generation failures stay in the accuracy denominator. Reference answers are loaded only
after generation. No hint, evidence extraction, learner attempt, or reference explanation is used.

A new folder retains per-question raw responses, predictions, failures, timing, settings and the scored
result. With Drive enabled, it is copied to **My Drive/IE4422/runs/answer_checks/<run-id>/**, including
partial files if the worker fails. One model serves the batch and is released when the worker exits.
Rerunning starts a new folder and loads a fresh model. The UI is stopped first to release its GPU memory.


In [ ]:
CHECK_LIMIT = 10  # @param {type:"integer"}
if INTERFACE == 'answer checker':
    if CHECK_LIMIT < 0:
        raise ValueError('CHECK_LIMIT must be positive, or 0 for all audited training questions.')
    if globals().get('UI_SERVER') is not None:
        UI_SERVER.stop()
    from datetime import datetime, timezone
    from notebooks.colab_support import complete_checkpoint
    config = read_json(WORK / 'notebooks/colab_config.json')
    checkpoint = WORK / 'models' / config['model_revision']
    if not complete_checkpoint(checkpoint, config):
        raise RuntimeError('Run the fixed-checkpoint download/restore cell first.')
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    ANSWER_CHECK_RUN = WORK / 'outputs/answer_checks' / run_id
    command = [str(PYTHON), str(WORK / 'check_scienceqa_answers.py'),
               '--model-path', str(checkpoint), '--output-dir', str(ANSWER_CHECK_RUN)]
    if CHECK_LIMIT:
        command.extend(['--limit', str(CHECK_LIMIT)])
    try:
        run_visible(command, cwd=WORK)
    finally:
        if PERSISTENT and ANSWER_CHECK_RUN.is_dir():
            saved = PERSISTENT / 'runs/answer_checks' / run_id
            shutil.copytree(ANSWER_CHECK_RUN, saved, dirs_exist_ok=True)
            print('Drive results:', saved)
    result = read_json(ANSWER_CHECK_RUN / 'result.json')
    print('Training answer-check summary:', result['summary'])
    print('Local results:', ANSWER_CHECK_RUN)
else:
    print('Skipped: select answer checker in step 3 to run this training-only check.')


## 9. Start browser UI (optional)

With `INTERFACE='browser UI'`, this cell starts a background server in the project environment, checks that it is ready, and embeds it below.
Use **I'm stuck — give me a hint** or submit an answer and reasoning with **Check my answer**.
Only train/validation are available, including through the server API.

The first hint/check loads Qwen into GPU memory; subsequent questions reuse that instance.
Rerunning this cell reuses the server. Set `RESTART_UI=True` only to deliberately stop it and start fresh;
reset the setting to False afterwards. Do not restart while a request is running.
Drive saving is synchronous: a job is marked complete only after its result has also been copied to Drive.
Keep the notebook open. Access uses Colab's own port proxy, without a public tunnel or extra account.


In [ ]:
UI_PORT = 8765  # @param {type:"integer"}
RESTART_UI = False  # @param {type:"boolean"}
if INTERFACE == 'browser UI':
    from google.colab import output
    from urllib.parse import urlsplit
    import importlib
    import notebooks.colab_ui as colab_ui
    importlib.reload(colab_ui)
    if globals().get('UI_SERVER') is None:
        UI_SERVER = colab_ui.NotebookUIServer()
    proxy_url = output.eval_js(f'google.colab.kernel.proxyPort({int(UI_PORT)}, {{cache: false}})')
    parsed_proxy = urlsplit(proxy_url)
    if parsed_proxy.scheme != 'https' or not parsed_proxy.netloc:
        raise RuntimeError('Colab did not return an HTTPS port proxy. Reconnect and rerun this cell.')
    proxy_origin = f'{parsed_proxy.scheme}://{parsed_proxy.netloc}'
    output_origin = output.eval_js('window.location.origin')
    status = UI_SERVER.start(
        PYTHON, WORK, port=int(UI_PORT), proxy_origin=proxy_origin,
        frame_ancestors=('https://colab.research.google.com', output_origin),
        persistent_output=PERSISTENT / 'runs/ui' if PERSISTENT else None,
        restart=RESTART_UI,
    )
    print('Server ready. Model:', 'loaded and reusable' if status['model_loaded'] else 'loads on the first hint/check')
    print('Results:', PERSISTENT / 'runs/ui' if PERSISTENT else WORK / 'outputs')
    print('Server log:', UI_SERVER.log_path)
    output.serve_kernel_port_as_iframe(int(UI_PORT), height=1100, cache_in_notebook=False)
else:
    if globals().get('UI_SERVER') is not None:
        UI_SERVER.stop()
    print('Browser UI disabled for the selected workflow.')


## 10. Check server status and logs (optional)

Rerun this cell if loading or generation fails. A ready server does not imply the model has loaded yet.
Logs and job status show model-load/generation failures. Saved results survive server restarts.


In [ ]:
if globals().get('UI_SERVER') is not None:
    if UI_SERVER.running:
        print(UI_SERVER.status())
    else:
        print('UI server is stopped.')
    print(UI_SERVER.logs())
else:
    print('UI has not been started in this session.')


## 11. Stop the UI when finished (optional)

Set `STOP_UI=True` and run this cell to stop the server and release its model memory.
It interrupts active jobs. Completed JSON results remain locally and, when enabled, in Drive.
Leave False while using the UI. To start again, rerun step 9.


In [ ]:
STOP_UI = False  # @param {type:"boolean"}
if STOP_UI and globals().get('UI_SERVER') is not None:
    UI_SERVER.stop()
    print('UI stopped; its model memory has been released.')
else:
    print('No stop requested. Set STOP_UI=True when finished with the browser UI.')


## Optional notebook demo: inspect an audited development question

These demo cells run only when `INTERFACE='notebook demo'` in step 3. They are skipped for the other workflows.
Leave `ITEM_ID` empty to use the first retained question from train/validation. Excluded and held-out test IDs are rejected.


In [ ]:
if globals().get('INTERFACE', 'notebook demo') == 'notebook demo':
    SPLIT = 'validation'  # @param ["train", "validation"]
    ITEM_ID = ''  # @param {type:"string"}
    if SPLIT not in ('train', 'validation'):
        raise ValueError('This development notebook does not run on held-out test items.')
    from run_scienceqa_tutor import load_question
    if not ITEM_ID.strip():
        ITEM_ID = read_json(WORK / f'manifests/scienceqa/{SPLIT}_final_ids.json')[0]
    item = load_question(WORK, SPLIT, ITEM_ID.strip(), data_source="final")
    print(item['item_id'], '\n', item['question'])
    if item['context'].strip():
        print('Supplied context:', item['context'])
    for choice in item['choices']:
        print(f"{choice['id']}. {choice['text']}")
    from IPython.display import Image, display
    display(Image(filename=item['image_path']))
else:
    print('Skipped: select notebook demo in step 3 to use this cell.')


### Enter the student's response (notebook demo only)

Enter an actual attempt for the displayed question. This optional demo uses response diagnosis and produces three hints.


In [ ]:
if globals().get('INTERFACE', 'notebook demo') == 'notebook demo':
    STUDENT_ANSWER = input('Student answer (option letter or text): ').strip()
    STUDENT_EXPLANATION = input('Student reasoning: ').strip()
    HINT_LEVELS = [1, 2, 3]
    if not STUDENT_ANSWER or not STUDENT_EXPLANATION:
        raise ValueError('Both a student response and reasoning are required.')
    ATTEMPT_FILE = WORK / 'tmp/student_attempt.json'
    write_json(ATTEMPT_FILE, {'answer': STUDENT_ANSWER, 'explanation': STUDENT_EXPLANATION})
else:
    print('Skipped: select notebook demo in step 3 to use this cell.')


### Generate and save three hints (notebook demo only)

The worker uses one model instance and reuses raw evidence/diagnosis strings across this item's three hint levels.
It exits afterwards and releases GPU memory. The UI server is stopped first to avoid loading two models.
Each completed stage is saved locally and, if enabled, to Drive. Package/GPU versions are recorded for demo runs.
This is not a validated evidence cache, hint-history protocol or A-E evaluation.


In [ ]:
if globals().get('INTERFACE', 'notebook demo') == 'notebook demo':
    if globals().get('UI_SERVER') is not None:
        UI_SERVER.stop()
    RUNS = WORK / 'outputs/colab'
    previous_runs = set(RUNS.glob('*')) if RUNS.exists() else set()
    command = [str(PYTHON), str(WORK / 'notebooks/run_demo.py'), '--split', SPLIT,
               '--item-id', item['item_id'], '--attempt', str(ATTEMPT_FILE),
               '--levels', *map(str, HINT_LEVELS), '--output-dir', str(RUNS)]
    if PERSISTENT:
        command.extend(['--persistent', str(PERSISTENT / 'runs')])
    run_visible(command, cwd=WORK)
    new_runs = set(RUNS.glob('*')) - previous_runs
    if len(new_runs) != 1:
        raise RuntimeError('Expected one new run directory; inspect outputs/colab.')
    LATEST_RUN = new_runs.pop()
    result = read_json(LATEST_RUN / 'result.json')
    run_record = read_json(LATEST_RUN / 'run.json')
    if run_record['status'] != 'complete':
        raise RuntimeError('This run did not complete; inspect the saved partial outputs.')
    for level, hint in result['hints'].items():
        print(f'\nHint {level}: {hint}')
    print('\nLocal results:', LATEST_RUN)
    if PERSISTENT:
        print('Drive results:', PERSISTENT / 'runs' / LATEST_RUN.name)
    print('Generation seconds:', round(run_record['generation_seconds'], 2))
else:
    print('Skipped: select notebook demo in step 3 to use this cell.')


## 12. Download results (optional)

Set `DOWNLOAD_RESULTS=True` to download the latest answer-check run, local UI results, or the latest
notebook-demo run as a ZIP. Drive already retains the answer-check folder when enabled, including
partial files after failure, under **My Drive/IE4422/runs/answer_checks/**. UI results are saved under
**My Drive/IE4422/runs/ui/**. Exports contain saved results and settings, without data or model weights.


In [ ]:
DOWNLOAD_RESULTS = False  # @param {type:"boolean"}
if DOWNLOAD_RESULTS:
    from google.colab import files
    export = WORK / 'tmp/ie4422_results.zip'
    export.parent.mkdir(parents=True, exist_ok=True)
    if INTERFACE == 'answer checker':
        if 'ANSWER_CHECK_RUN' not in globals() or not ANSWER_CHECK_RUN.is_dir():
            raise RuntimeError('Run the training answer-check cell first.')
        export = Path(shutil.make_archive(str(WORK / 'tmp' / ANSWER_CHECK_RUN.name),
                                          'zip', root_dir=ANSWER_CHECK_RUN))
    elif INTERFACE == 'browser UI':
        results = sorted((WORK / 'outputs').glob('ui_*.json'))
        if not results:
            raise RuntimeError('No saved UI results yet. Complete a hint or answer check first.')
        with zipfile.ZipFile(export, 'w', zipfile.ZIP_DEFLATED) as archive:
            for result_path in results:
                archive.write(result_path, result_path.name)
    else:
        if 'LATEST_RUN' not in globals():
            raise RuntimeError('Run the notebook demo generation cell first.')
        export = Path(shutil.make_archive(str(WORK / 'tmp' / LATEST_RUN.name), 'zip', root_dir=LATEST_RUN))
    files.download(str(export))
else:
    print('Download disabled. Drive saving, when enabled, happens automatically.')


For another answer-only batch, rerun step 8. It always uses the audited training split.
Use train results for development; reserve validation/test evaluation for a later agreed protocol.

The UI's first request loads the model; later requests reuse it. Rerun **Start browser UI** with
`RESTART_UI=False` to redisplay it without loading another model. Use `RESTART_UI=True` after changing server settings.
Stopping/restarting loses in-memory jobs and the loaded model, while saved result files remain.
To recover after runtime deletion, rerun setup and restore data/weights from Drive.

For the optional demo, choose `notebook demo` in step 3 and run its three cells. The UI must be stopped first;
the generation cell also enforces this. Rebuild/reupload both distribution files after editing source.

Sources: [Colab limits and web-UI restrictions](https://research.google.com/colaboratory/faq.html),
[Colab's iframe port helper](https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_util.py),
[Hugging Face checkpoint downloads](https://huggingface.co/docs/huggingface_hub/guides/download).
Actual Colab proxy/browser behavior and CUDA inference still require verification on the assigned runtime.
